In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import xgboost as xgb

# ==========================================
# 1. LOAD DATA & SPLIT
# ==========================================
data_path = '../modelData/master_modeling_matrix.csv'
df = pd.read_csv(data_path, index_col='loan_id')
df.index = df.index.astype(str)

cols_to_drop = ['target', 'status', 'is_closed', 'loan_date', 'account_id']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

X = df[feature_cols]
y = df['target']

# Standard 80/20 train-test split for record inspection
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale weight for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# ==========================================
# 2. TRAIN BOTH MODELS
# ==========================================
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

model_xgb = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1,
)
model_xgb.fit(X_train, y_train)
xgb_probs = model_xgb.predict_proba(X_test)[:, 1]

# ==========================================
# 3. BUILD SIDE-BY-SIDE COMPARISON TABLE
# ==========================================
comparison_df = pd.DataFrame(
    {
        'Actual_Target': y_test.values,
        'RF_Default_Prob': rf_probs,
        'XGB_Default_Prob': xgb_probs,
    },
    index=X_test.index,
)

# Apply a 0.5 threshold to see binary predictions
comparison_df['RF_Pred'] = (comparison_df['RF_Default_Prob'] >= 0.5).astype(int)
comparison_df['XGB_Pred'] = (comparison_df['XGB_Default_Prob'] >= 0.5).astype(
    int
)

print('=' * 75)
print(' RECORD-BY-RECORD MODEL COMPARISON (TEST SET SAMPLE)')
print('=' * 75)
# Configure Pandas to show all rows and columns without truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# Print the complete dataframe
print(comparison_df)

# Reset options back to default afterwards (optional)
pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")
print('=' * 75)

# Highlight specific insights
matches = (
    comparison_df['RF_Pred'] == comparison_df['XGB_Pred']
).mean() * 100
print(
    f'Percentage where Random Forest and XGBoost made the exact same binary prediction: {matches:.2f}%'
)

# Show actual defaults caught by both models
defaults_caught = comparison_df[
    (comparison_df['Actual_Target'] == 1)
    & (comparison_df['RF_Pred'] == 1)
    & (comparison_df['XGB_Pred'] == 1)
]
print(
    f'\nTotal actual defaults in test set: {(y_test == 1).sum()}'
)
print(
    f'Defaults correctly flagged by BOTH models: {len(defaults_caught)}'
)

 RECORD-BY-RECORD MODEL COMPARISON (TEST SET SAMPLE)
         Actual_Target  RF_Default_Prob  XGB_Default_Prob  RF_Pred  XGB_Pred
loan_id                                                                     
5138                 0         0.113852          0.056991        0         0
5382                 0         0.079224          0.018619        0         0
6934                 0         0.114032          0.007490        0         0
5526                 0         0.012348          0.004793        0         0
6820                 0         0.126035          0.021649        0         0
7129                 0         0.029759          0.008227        0         0
5865                 0         0.017335          0.014751        0         0
6546                 0         0.008931          0.004450        0         0
6306                 0         0.253225          0.081946        0         0
5388                 0         0.212830          0.054070        0         0
4973                 0 

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# Use XGBoost probabilities from our comparison dataframe
probs = comparison_df['XGB_Default_Prob'].values
true_y = comparison_df['Actual_Target'].values

# Define your target custom threshold (e.g., 0.20 instead of 0.50)
custom_threshold = 0.20

# Generate new predictions based on the lower threshold
new_preds = (probs >= custom_threshold).astype(int)

# Build confusion matrix
cm = confusion_matrix(true_y, new_preds)
tn, fp, fn, tp = cm.ravel()

print("=" * 60)
print(f" BUSINESS IMPACT ANALYSIS AT THRESHOLD = {custom_threshold}")
print("=" * 60)
print(f" Total Actual Defaults in Test Set: {tp + fn}")
print(f" Defaults Successfully Caught (True Positives): {tp}")
print(f" Defaults Missed / Bank Losses (False Negatives): {fn}")
print(f" Safe Loans Correctly Approved (True Negatives): {tn}")
print(f" Safe Loans Flagged for Review (False Positives): {fp}")
print("=" * 60)

# Compare with the default 0.5 threshold performance
old_preds = (probs >= 0.50).astype(int)
old_cm = confusion_matrix(true_y, old_preds)
_, _, old_fn, old_tp = old_cm.ravel()

print(f"Comparison vs 0.50 Threshold:")
print(f" - Defaults caught went from {old_tp} -> {tp} (+{tp - old_tp} defaults saved!)")
print(f" - Missed defaults dropped from {old_fn} -> {fn} (-{old_fn - fn} fewer bank write-offs)")
print("=" * 60)

 BUSINESS IMPACT ANALYSIS AT THRESHOLD = 0.2
 Total Actual Defaults in Test Set: 15
 Defaults Successfully Caught (True Positives): 12
 Defaults Missed / Bank Losses (False Negatives): 3
 Safe Loans Correctly Approved (True Negatives): 116
 Safe Loans Flagged for Review (False Positives): 6
Comparison vs 0.50 Threshold:
 - Defaults caught went from 9 -> 12 (+3 defaults saved!)
 - Missed defaults dropped from 6 -> 3 (-3 fewer bank write-offs)


In [3]:
from sklearn.metrics import precision_recall_curve, confusion_matrix
import numpy as np

# Extract probabilities and actual targets from your comparison dataframe
probs = comparison_df['XGB_Default_Prob'].values
true_y = comparison_df['Actual_Target'].values

# Calculate precision, recall, and thresholds across the curve
precisions, recalls, thresholds = precision_recall_curve(true_y, probs)

# Set beta > 1 to prioritize Recall over Precision (F2 score weights recall 2x more)
beta = 2.0
f_beta_scores = (
    (1 + beta**2)
    * (precisions * recalls)
    / ((beta**2 * precisions) + recalls + 1e-10)
)

# Find the threshold that maximizes the F2 score
best_idx = np.argmax(f_beta_scores)
optimal_threshold = (
    thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
)

print('=' * 60)
print(f' OPTIMAL RECALL-WEIGHTED THRESHOLD (F2 Score): {optimal_threshold:.4f}')
print('=' * 60)

# Evaluate performance at this new mathematically optimized threshold
opt_preds = (probs >= optimal_threshold).astype(int)
cm = confusion_matrix(true_y, opt_preds)
tn, fp, fn, tp = cm.ravel()

print(f'Confusion Matrix at F2 Threshold ({optimal_threshold:.4f}):')
print(cm)
print(f'\nResults Breakdown:')
print(f' - Defaults Caught (True Positives): {tp} / {tp + fn}')
print(f' - Missed Defaults (False Negatives): {fn}')
print(f' - False Alarms / Extra Reviews (False Positives): {fp}')
print('=' * 60)

 OPTIMAL RECALL-WEIGHTED THRESHOLD (F2 Score): 0.2466
Confusion Matrix at F2 Threshold (0.2466):
[[117   5]
 [  3  12]]

Results Breakdown:
 - Defaults Caught (True Positives): 12 / 15
 - Missed Defaults (False Negatives): 3
 - False Alarms / Extra Reviews (False Positives): 5
